In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  

import json

import torch

import matplotlib.pyplot as plt
from helpers.utils import set_seed
from helpers.viz import draw_yolo_boxes

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.__version__, device)



2.12.0+cu130 cuda


In [2]:
from eytnet.config import Config
cfg = Config.load("configs/experiment.json")

print(json.dumps(cfg.raw, indent=2))
print("grid :", cfg.grid_sizes)
print("anchor:", cfg.anchors)

{
  "experiment_name": "eytnet_baseline",
  "data_root": "data",
  "class_names": [
    "fire",
    "smoke"
  ],
  "image_size": 640,
  "strides": [
    16,
    32
  ],
  "anchors_path": "configs/anchors.json",
  "seed": 42,
  "batch_size": 16,
  "num_workers": 4,
  "epochs": 30,
  "optimizer": "adam",
  "learning_rate": 0.001,
  "momentum": 0.9,
  "weight_decay": 0.0005,
  "lr_scheduler": "cosine",
  "final_lr_factor": 0.05,
  "warmup_epochs": 3,
  "augmentation_enabled": true,
  "use_clahe": false,
  "use_amp": true,
  "gradient_clip_norm": 10.0,
  "early_stopping_patience": 12,
  "lambda_box": 5.0,
  "lambda_objectness": 1.0,
  "lambda_class": 1.0,
  "negative_objectness_weight": 0.5,
  "ignore_iou_threshold": 0.5,
  "decode_score_floor": 0.01,
  "validation_score_threshold": 0.25,
  "nms_iou_threshold": 0.5,
  "max_detections": 300,
  "run_root": "models/eytnet"
}
grid : [40, 20]
anchor: [[(9.5, 11.0), (21.0, 22.5), (36.0, 48.0)], [(69.5, 80.0), (110.5, 148.5), (241.50000000000003,

In [ ]:
from eytnet.dataset import FireDetectionDataset, build_dataloader
ds = FireDetectionDataset(cfg.data_root, "train", cfg.image_size, augment=True)

img, tgt, meta = ds[0]

print(len(ds), img.shape, img.dtype, img.min().item(), img.max().item())
print(tgt)

3900 torch.Size([3, 640, 640]) torch.float32 0.0 0.8235294222831726
tensor([[0.0000, 0.6211, 0.5987, 0.0636, 0.0531]])


In [5]:
from eytnet.model import build_model

model = build_model(cfg)
dummy = torch.randn(2,3,640,640)

outs = model(dummy)

for o, s in zip(outs, cfg.strides):
    print(s, tuple(o.shape))
print("parametre:", f"{model.count_parameters():,}")


16 (2, 3, 40, 40, 7)
32 (2, 3, 20, 20, 7)
parametre: 7,208,138
